In [1]:
%pip install transformers torch pandas scikit-learn accelerate

  Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import re
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, 
    DistilBertForSequenceClassification, 
    Trainer, 
    TrainingArguments
)
from huggingface_hub import login

In [ ]:
# initally ran this in google colab so saved the token and output dir there
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    
    # Get token from Colab Secrets
    hf_token = userdata.get('HF_TOKEN')
    
    OUTPUT_DIR = "/content/drive/MyDrive/Transaction_Model" 

except ImportError:
    # For local, get token from environment variable or paste it manually
    hf_token = os.getenv('HF_TOKEN') 
    
    # Save in the current directory locally
    OUTPUT_DIR = "./model_output"

# Login to HuggingFace
if hf_token:
    login(hf_token)
else:
    print("Warning: No HF_TOKEN found. You may need to login manually.")

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# custom dataset class used to handle our transaction data
class TransactionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

def clean_description(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'\s*#\d+', '', text)    
    text = re.sub(r'[a-z]+\d+', '', text)  
    text = re.sub(r'\((.*?)\)', '', text)  
    text = re.sub(r'\s*-.*', '', text)     
    text = text.strip()
    return text

In [ ]:
df = pd.read_parquet("hf://datasets/mitulshah/transaction-categorization/default/train/0000.parquet")

df_usa = df[df['country'] == "USA"].copy()
df_usa = df_usa[['transaction_description', 'category']]

df_subset = df_usa.groupby("category").apply(lambda x: x.sample(n=min(len(x), 7500))).reset_index(drop=True)

print(f"Original size: {len(df)} rows")
print(f"New balanced size: {len(df_subset)} rows")

df_subset['clean_description'] = df_subset['transaction_description'].apply(clean_description)

texts = df_subset['clean_description'].tolist()
labels = df_subset['category'].tolist()

# used to map our labels with numbers so the model output a single number, which we later translate
unique_labels = df['label'].unique().tolist()

id2label = {i: label for i, label in enumerate(unique_labels)} # what the model uses to tell use the answer
label2id = {label: i for i, label in enumerate(unique_labels)} # what i use to tell the model the answer

labels_numeric = [label2id[label] for label in labels]

# Split Data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels_numeric, test_size=0.2, random_state=42
)

Mounted at /content/drive


/tmp/ipython-input-502638780.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_subset = df_usa.groupby("category").apply(lambda x: x.sample(n=min(len(x), 7500))).reset_index(drop=True)


Original size: 4501043 rows
New balanced size: 75000 rows
category
Charity & Donations           7500
Entertainment & Recreation    7500
Financial Services            7500
Food & Dining                 7500
Government & Legal            7500
Healthcare & Medical          7500
Income                        7500
Shopping & Retail             7500
Transportation                7500
Utilities & Services          7500
Name: count, dtype: int64
                transaction_description            clean_description
57973                            Macy's                       macy's
13072            Gym #5757 - USA Campus                          gym
43845     Children's Hospital TXN864782          children's hospital
13793                       Escape Room                  escape room
2169         Mosque - USA - Dinner Time                       mosque
71069                Spectrum - Morning                     spectrum
9779                  Theater TXN950401                      theater
52659

In [ ]:
class TransactionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
# used to convert text into tokens and convert them into id numbers
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# corresponding tokenizer for training and validation. truncation and padding are there to
# keep the math clean
train_encodings = tokenizer(
    train_texts,
    padding='max_length',
    truncation=True,
    max_length=42
)

val_encodings = tokenizer(
    val_texts,
    padding='max_length',
    truncation=True,
    max_length=42
)

# Create Datasets
train_dataset = TransactionDataset(train_encodings, train_labels)
val_dataset = TransactionDataset(val_encodings, val_labels)

# load the model, already knows english and now we are fine tuning it to our specific purpose
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

# rules for the model
training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# engine to run it
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

In [ ]:
# Save the final model and tokenizer to your output directory
save_path = f"{OUTPUT_DIR}/final_model"
print(f"Saving final model to {save_path}...")

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.027400,0.031675
2,0.022900,0.024397
3,0.021200,0.024237


Saving model to ./transaction_classification_model...


('./transaction_classification_model/tokenizer_config.json',
 './transaction_classification_model/special_tokens_map.json',
 './transaction_classification_model/vocab.txt',
 './transaction_classification_model/added_tokens.json',
 './transaction_classification_model/tokenizer.json')

In [ ]:
!zip -r transaction_classification_model.zip transaction_classification_model

	zip warning: name not matched: transaction_classification_model

zip error: Nothing to do! (try: zip -r transaction_classification_model.zip . -i transaction_classification_model)
